# Four-way model comparison (apples-to-apples)

Compare four ways of training the same gap-fill U-Net on the **same region, same channels, same stats,
same hyperparameters**, so the only variables are *streaming vs not* and *spatial patches vs whole domain*.

| Run | What | Source | Patch (lat,lon) | Notes |
|---|---|---|---|---|
| 1 | new code, whole-domain | IO_rechunked | full grid | should reproduce run 3 |
| 2 | new code, spatial patches | IO_rechunked | 40 x 56 | the approach under test |
| 3 | old streaming notebook | IO.zarr | full grid | run in 2-U_Net-Streaming |
| 4 | original non-streaming Fit | shared std zarr | full grid | run in 2-U-Net_Fit |

**Held constant for fairness:** `features=[]` (9 channels), 3yr train / 1yr val / 1yr test, one shared
`stats` object, batch size **1** (whole-domain runs OOM the GPU above 1), same U-Net, same optimizer,
same EarlyStopping. Copernicus L4 truth (`CHL_cmes-gapfree`) is pulled from the original `IO.zarr` (it is
not in the rechunked file).

**UNTESTED:** the TF/training/plotting cells were written but not run here (no Hub/GPU). Treat the first
execution as the real test. This will take several hours; runs 3 and 4 happen in their own notebooks.

In [ ]:
!pip install git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@troy-branch

In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd, xarray as xr, dask.array as da
import tensorflow as tf
from tensorflow.keras import Input, layers
from tensorflow.keras.callbacks import EarlyStopping
from xbatcher import BatchGenerator
import matplotlib.pyplot as plt
import mindthegap as mtg

gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", gpus)

## 1. Config (shared by all runs)

Region is the full grid; if a single whole-domain frame OOMs the GPU at batch 1 (runs 1/3/4), shrink it
here (e.g. the Arab Sea box) and rerun. Keep it the SAME for all four.

In [ ]:
RECHUNKED = "/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr"
ORIGINAL  = os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr")

# Region: full grid. To shrink, set a lat/lon subset here and it applies everywhere.
SUBSET = None          # or dict(lat=slice(31,5), lon=slice(42,80)) for the Arab Sea

features = []                                   # chlorophyll-derived channels only (PACE-like)
train_year, train_range, val_range, test_range = 2015, 3, 1, 1
BATCH_SIZE = 1                                  # whole-domain runs cap here; keep equal for all
DAY_BATCH  = 100                                # days per read for the streaming runs
LAT_PATCH_FULL = LON_PATCH_FULL = None          # filled in after we know the grid size

MODEL_DIR = "models/compare"
os.makedirs(MODEL_DIR, exist_ok=True)
SHARED_ZARR = f"{MODEL_DIR}/shared_std.zarr"    # standardized data for run 4 (Fit)

def open_region(path):
    ds = xr.open_zarr(path, chunks={})
    if SUBSET is not None:
        ds = ds.sel(**SUBSET)
    ds = mtg.crop_to_multiple(ds, multiple=8)
    # keep only the train..test window so we never build over the full record
    t_lo = f'{train_year}-01-01'
    t_hi = f'{train_year+train_range+val_range+test_range}-01-01'
    return ds.sel(time=slice(t_lo, t_hi))

## 2. Shared stats (computed once, from the original for speed)

All four runs standardize with THIS one stats object, so the inputs are identical. We compute it on the
original `IO.zarr` (fast `time=100` reads); the values are identical to the rechunked file.

In [ ]:
orig = open_region(ORIGINAL)
LAT_PATCH_FULL, LON_PATCH_FULL = orig.sizes['lat'], orig.sizes['lon']
print("grid:", dict(orig.sizes), " full patch:", (LAT_PATCH_FULL, LON_PATCH_FULL))

# compute stats once (features=[] -> only the CHL-derived channel stats)
_, SHARED_STATS = mtg.build_standardized_lazy(orig, features, train_year, train_range,
                                              standardize_chl=True)
print("shared stats channels:", list(SHARED_STATS['feat_stats'].keys()))

## 3. Shared standardized zarr (for run 4, the non-streaming Fit)

Write the standardized dataset once, with the shared stats, so run 4 loads exactly the same channels the
streaming runs build on the fly. Whole-domain chunks since Fit loads it all into RAM.

In [ ]:
ds_std_full, _ = mtg.build_standardized_lazy(
    orig, features, train_year, train_range, standardize_chl=True,
    stats=SHARED_STATS, output_chunks={"time": 100, "lat": -1, "lon": -1},
)
# write once (skip if it already exists)
if not os.path.exists(SHARED_ZARR):
    ds_std_full.to_zarr(SHARED_ZARR, mode="w")
    print("wrote", SHARED_ZARR)
else:
    print("exists:", SHARED_ZARR)
X_VARS = [v for v in ds_std_full.data_vars if v != 'CHL']
NUM_FEATURES = len(X_VARS)
print("channels:", NUM_FEATURES, X_VARS)

## 4. Shared U-Net (fully convolutional)

In [ ]:
def UNet(num_features):
    inputs = Input(shape=(None, None, num_features))
    x = inputs
    filters = [64, 128, 256]
    ec = []
    for f in filters:
        ec.append(x)
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.BatchNormalization()(x)
    for f, e in zip(filters[:-1][::-1], ec[::-1][:-1]):
        x = layers.Conv2DTranspose(f, 3, 2, padding='same')(x)
        x = layers.concatenate([x, e])
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(f, 3, 2, padding='same')(x)
    x = layers.concatenate([x, ec[0]])
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
    out = layers.Conv2D(1, 3, padding='same', activation='linear')(x)
    m = tf.keras.Model(inputs, out, name='U-net')
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

## 5. Runs 1 and 2 (streaming, new code)

Both stream from the rechunked file with the shared stats; run 1 uses a whole-domain patch, run 2 uses
40 x 56. The training window is cached in RAM (`.load()`), and batch size is 1 for parity with the old
runs.

In [ ]:
def train_streaming(source_path, lat_patch, lon_patch, model_name):
    ds = open_region(source_path)
    ds_std, _ = mtg.build_standardized_lazy(
        ds, features, train_year, train_range, standardize_chl=True,
        stats=SHARED_STATS, output_chunks={"time": DAY_BATCH, "lat": lat_patch, "lon": lon_patch},
    )
    x_vars = [v for v in ds_std.data_vars if v != 'CHL']
    sig = (tf.TensorSpec((lat_patch, lon_patch, len(x_vars)), tf.float32),
           tf.TensorSpec((lat_patch, lon_patch, 1), tf.float32))
    dims = {"time": DAY_BATCH, "lat": lat_patch, "lon": lon_patch}
    ov   = {"time": 0, "lat": 0, "lon": 0}

    dtr = ds_std.sel(time=slice(f'{train_year}-01-01', f'{train_year+train_range}-01-01')).load()
    dva = ds_std.sel(time=slice(f'{train_year+train_range}-01-01', f'{train_year+train_range+val_range}-01-01')).load()
    btr = BatchGenerator(dtr, input_dims=dims, input_overlap=ov)
    bva = BatchGenerator(dva, input_dims=dims, input_overlap=ov)

    tr = tf.data.Dataset.from_generator(mtg.make_tf_gen(btr, x_vars), output_signature=sig
        ).shuffle(512).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
    va = tf.data.Dataset.from_generator(mtg.make_tf_gen(bva, x_vars), output_signature=sig
        ).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
    steps_tr = (len(btr) * DAY_BATCH) // BATCH_SIZE
    steps_va = (len(bva) * DAY_BATCH) // BATCH_SIZE

    model = UNet(len(x_vars))
    es = EarlyStopping(patience=10, restore_best_weights=True)
    hist = model.fit(tr, epochs=50, steps_per_epoch=steps_tr,
                     validation_data=va, validation_steps=steps_va, callbacks=[es])
    model.save(f"{MODEL_DIR}/{model_name}.keras")
    return model, hist

In [ ]:
# Run 1: whole-domain via the new code (should match run 3)
m1, h1 = train_streaming(RECHUNKED, LAT_PATCH_FULL, LON_PATCH_FULL, "run1_newcode_fullgrid")

In [ ]:
# Run 2: spatial 40x56 patches (the approach under test)
m2, h2 = train_streaming(RECHUNKED, 40, 56, "run2_spatial_40x56")

## 6. Runs 3 and 4 (their own notebooks)

Run these in `2-U_Net-Streaming_Version` (run 3) and `2-U-Net_Fit` (run 4) with the SAME settings, and
save into `models/compare/` so section 7 can load them:

- **Both:** `features = []`, `train_year=2015, train_range=3, val_range=1, test_range=1`, batch size **1**,
  same region (`SUBSET` above), epochs 50, EarlyStopping patience 10.
- **Run 3 (streaming):** open `IO.zarr` (original), pass `stats=SHARED_STATS` to `build_standardized_lazy`
  (paste the printed dict), save as `models/compare/run3_streaming.keras`.
- **Run 4 (Fit):** load `SHARED_ZARR` (written in section 3) instead of the GCS Arab Sea zarr, then
  `data_split` -> numpy -> train, save as `models/compare/run4_fit.keras`.

Reusing the shared stats and shared zarr is what keeps run 3 and run 4 on identical inputs to runs 1/2.

## 7. Evaluation: per-day MAE vs Copernicus across the test year

For each model, over the test year, predict each day and score it against the Copernicus L4 gapfree
product on the pixels that were actually gaps (where level-3 is missing). All four on one plot. Same
region and channels for all, so it is apples-to-apples.

In [ ]:
from mindthegap import compute_mae

TEST_YEAR = train_year + train_range + val_range   # 2019 for the default config

# channels for the test year (shared; built once from the original)
orig_full = xr.open_zarr(ORIGINAL, chunks={})
if SUBSET is not None:
    orig_full = orig_full.sel(**SUBSET)
orig_full = mtg.crop_to_multiple(orig_full, multiple=8)
ds_std_test, _ = mtg.build_standardized_lazy(
    orig_full.sel(time=str(TEST_YEAR)), features, train_year, train_range,
    standardize_chl=True, stats=SHARED_STATS)
y_mean, y_std = SHARED_STATS['CHL'][0], SHARED_STATS['CHL'][1]
Xtest = np.stack([np.nan_to_num(ds_std_test[v].values, nan=0.0) for v in X_VARS], axis=-1).astype(np.float32)

def per_day_mae(model):
    pred = model.predict(Xtest, batch_size=4, verbose=0)[..., 0] * y_std + y_mean
    gap = np.log(orig_full.sel(time=str(TEST_YEAR))['CHL_cmes-gapfree'].values)
    lvl3 = orig_full.sel(time=str(TEST_YEAR))['CHL_cmes-level3'].values
    pred = np.where(~np.isnan(lvl3), np.nan, pred)          # score only the gap pixels
    return [compute_mae(t, p) for t, p in zip(gap, pred)]

In [ ]:
paths = {
    "1 new/full":   f"{MODEL_DIR}/run1_newcode_fullgrid.keras",
    "2 spatial":    f"{MODEL_DIR}/run2_spatial_40x56.keras",
    "3 streaming":  f"{MODEL_DIR}/run3_streaming.keras",
    "4 Fit":        f"{MODEL_DIR}/run4_fit.keras",
}
plt.figure(figsize=(11, 5))
for label, p in paths.items():
    if not os.path.exists(p):
        print("missing (run it first):", p); continue
    mae = per_day_mae(tf.keras.models.load_model(p))
    plt.plot(mae, label=f"{label}  (mean {np.nanmean(mae):.4f})")
plt.xlabel("day of test year"); plt.ylabel("MAE vs Copernicus L4 (log Chl-a)")
plt.title(f"Per-day gap-fill error, test year {TEST_YEAR}")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## Notes / open items

- **Region vs GPU:** if runs 1/3/4 OOM on a whole full-grid frame at batch 1, set `SUBSET` to a smaller
  box (e.g. Arab Sea) and rerun everything on that region.
- **Run 1 vs Run 3** are both whole-domain and should match closely; that is the "new code == old code"
  check. **Run 2 vs the others** is the real question: does 40x56 patch training match whole-domain?
- The per-day eval uses training-mode channels. A stricter gap-fill eval (feed dense level-3 as
  `masked_CHL`, mark real clouds as gaps) mirrors the notebooks' `yearly_MAD_vs_cloud`; we can swap that
  in once the four models exist.
- Add a few visual gapfill panels (same region/extent) for a couple of common dates for a qualitative look.